[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aditya-siraskar/PINN_FTRE/blob/main/notebooks/03_phase1_stage2c_ablations.ipynb)

# 03 · Stage 2c: ablations of the PINN design choices

**Phase:** Phase 1: validation (1-D)

**Goal:** Test each tail-specific design choice instead of assuming it helps (log residual, tail sampling, Gauss–Hermite nodes, normalisation).

**Outputs:** `configs/ablations/*.yaml`, `results/stage2_pinn1d_<tag>/`, `figures/stage2_ablation/fig01_ablation_summary`

> Runtime → Change runtime type → **T4 GPU**. Run the setup cell first. With `USE_DRIVE = True` the repo lives in
> `MyDrive/PINN_FTRE`, so figures and results persist. Core logic lives in `pinn_tailrisk/` and `scripts/`;
> this notebook only runs it and shows the results.

In [ ]:
# @title Setup: repo on Google Drive, install, GPU check  { display-mode: "form" }
USE_DRIVE = True   # keep figures/results across Colab sessions (recommended)
REPO_URL = "https://github.com/aditya-siraskar/PINN_FTRE"

import os, sys
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
else:
    BASE = "/content"
ROOT = f"{BASE}/PINN_FTRE"
if not os.path.exists(ROOT):
    !git clone -q {REPO_URL} "{ROOT}"
else:
    rc = os.system(f'git -C "{ROOT}" pull -q --ff-only')
    if rc != 0:   # usually: result files on Drive that have since been committed to GitHub
        print("WARNING: git pull failed; the Drive copy is out of date. To sync (keeps untracked files):
"
              f'  !git -C "{ROOT}" fetch -q origin && git -C "{ROOT}" reset -q --hard origin/main')
%cd "{ROOT}"
!pip install -q -e .
sys.path.insert(0, ROOT)

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (Runtime > Change runtime type > T4 GPU)")

In [ ]:
# autoreload picks up library edits without a restart. Colab's IPython build imports the
# removed `imp` module on Python >= 3.12, so it is optional: if it is skipped, use
# Runtime > Restart session after editing pinn_tailrisk/.
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except ModuleNotFoundError:
    print("autoreload unavailable here; restart the session after editing the library")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pinn_tailrisk import plotting as P
from pinn_tailrisk.notebook import show_figures, load_metrics
P.use_style()

## 1. Variants
Each variant changes **one** setting relative to the baseline `configs/stage2_pinn1d.yaml` (now `n_gh = 48`),
and each runs with **3 seeds**, so differences can be told apart from seed noise.
18 runs × ~9 min ≈ **2.7 h on a T4**. Set `SEEDS = [0]` and/or `STEPS = 5000` for a quick pass.

In [ ]:
import yaml, copy, pathlib
STEPS = 20000
SEEDS = [0, 1, 2]
base = yaml.safe_load(open("configs/stage2_pinn1d.yaml"))
VARIANTS = {                       # tag: change vs baseline
    "baseline":       {},
    "linear_resid":   {"residual": "linear"},
    "no_tail_sampl":  {"tail_share": 0.0},
    "gh8":            {"n_gh": 8},
    "gh24":           {"n_gh": 24},
    "no_norm":        {"w_norm": 0.0},
}
pathlib.Path("configs/ablations").mkdir(exist_ok=True)
for tag, change in VARIANTS.items():
    c = copy.deepcopy(base); c["pinn"].update(change)
    yaml.safe_dump(c, open(f"configs/ablations/{tag}.yaml", "w"))
print(list(VARIANTS), "x seeds", SEEDS)

## 2. Train all runs
Resumable: finished runs are saved to Drive and skipped on rerun, so a Colab disconnect only costs the run in progress.

In [ ]:
for tag in VARIANTS:
    for seed in SEEDS:
        run = f"ab2_{tag}_s{seed}"
        if pathlib.Path(f"results/stage2_pinn1d_{run}/metrics.json").exists():
            print("skip", run); continue
        !python scripts/stage2_pinn1d.py --config configs/ablations/{tag}.yaml --steps {STEPS} --seed {seed} --tag {run}

## 3. Summary figure
Bars are the mean over seeds; the black whiskers span min–max, and dots are the individual seeds.
A difference counts only if the whiskers do not overlap.

In [ ]:
rows = []
for tag in VARIANTS:
    for seed in SEEDS:
        run = f"stage2_pinn1d_ab2_{tag}_s{seed}"
        if not pathlib.Path(f"results/{run}/metrics.json").exists():
            continue
        m = load_metrics(run); r = pd.DataFrame(m["risk"])
        for h in (1, 10):
            q = r[(r.h == h) & (r.alpha == 0.99)].iloc[0]
            rows.append(dict(variant=tag, seed=seed, horizon=f"{h}-day", es99_abs_rel=abs(q.es_rel_pinn),
                             var99_abs_rel=abs(q.var_rel_pinn),
                             tail_log_mae=m["density_error"][f"{h}d"]["left_tail_mae_log"], train_min=m["train_time_s"] / 60))
ab = pd.DataFrame(rows)
summary = ab.groupby(["variant", "horizon"])[["es99_abs_rel", "var99_abs_rel", "tail_log_mae", "train_min"]].agg(["mean", "min", "max"])
display(summary.style.format("{:.2e}"))
ab.to_csv("results/stage2_ablation_seeds.csv", index=False)

order = [v for v in VARIANTS if v in set(ab.variant)]
y = np.arange(len(order))
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8), sharey=True)
for ax, col, xl, title in [(axes[0], "es99_abs_rel", "|ES₉₉ relative error|", "Risk accuracy"),
                           (axes[1], "tail_log_mae", "left-tail mean |log error|", "Tail density accuracy")]:
    for h, c, dy in [("1-day", P.SEQ_BLUE(0.45), -0.17), ("10-day", P.SEQ_BLUE(0.95), 0.17)]:
        g = ab[ab.horizon == h].groupby("variant")[col]
        mean, lo, hi = g.mean().loc[order], g.min().loc[order], g.max().loc[order]
        ax.barh(y + dy, mean.values, 0.3, color=c, label=h)
        ax.errorbar(mean.values, y + dy, xerr=[mean - lo, hi - mean], fmt="none", ecolor=P.INK, elinewidth=0.9, capsize=2)
        for i, v in enumerate(order):
            pts = ab[(ab.horizon == h) & (ab.variant == v)][col]
            ax.plot(pts, np.full(len(pts), y[i] + dy), lw=0, marker="o", ms=2.5, color=P.INK)
    ax.set(xscale="log", xlabel=xl, title=title); ax.grid(axis="y", visible=False)
axes[0].axvline(0.01, color=P.AXIS, lw=0.8)
axes[0].set(yticks=y, yticklabels=order); axes[0].invert_yaxis()
axes[0].legend(title="horizon", loc="lower right"); P.panel_labels(axes); fig.tight_layout()
P.savefig(fig, "stage2_ablation", "fig01_ablation_summary")
show_figures("stage2_ablation")